# RoSE on ChartQAPro — **all MCQ questions**, headless Kaggle commit

Runs the repo's real pipeline (`chartqapro/rose_chartqapro.py` + `scoring.py`) over
**every Multi-Choice question in ChartQAPro**, then evaluates the run offline with
`ablate.py`, `summarize.py` and `export_csv.py` — so the notebook, the CLI tables and
the web dashboard all report the same numbers.

| | |
|---|---|
| **Dataset** | [`ahmed-masry/ChartQAPro`](https://huggingface.co/datasets/ahmed-masry/ChartQAPro) — `Multi Choice` only |
| **Model** | `Qwen/Qwen2.5-VL-7B-Instruct`, 4-bit |
| **Pipeline** | cloned from the repo at `REPO_REF` — no algorithm code is re-implemented here |
| **Scoring** | `scoring.py` only (single source of truth) |

## Run it headless on Kaggle

1. **Settings** → Accelerator **GPU T4 ×2** (or P100), Internet **ON**, Persistence *off*.
2. **Save Version → Save & Run All (Commit)**. Every cell is non-interactive and
   failure-tolerant, so the commit finishes and publishes its outputs even if the
   GPU wall is hit mid-run.
3. `TIME_BUDGET_MIN` (Cell 1) stops the loop *before* Kaggle kills the session and
   writes a complete checkpoint. `meta_mcq.json` records `"complete": false` when
   the run was cut short.

## Resuming a cut-short commit

Add the finished version's output as a dataset input
(**+ Add Data → Your Work → this notebook's output**), then set in Cell 1:

```python
RESUME_FROM = "/kaggle/input/<your-notebook-output>/results/checkpoint_mcq.json"
```

The pool is rebuilt from the checkpoint before the loop continues, exactly as
`rose_chartqapro.run()` does on a warm restart.

## Outputs (`/kaggle/working/results/`)

| File | What it is |
|---|---|
| `checkpoint_mcq.json` | resume point, rewritten every `CHECKPOINT_EVERY` questions |
| `rose_mcq.json` | the run — one record per question, **including every sampled path** |
| `meta_mcq.json` | model, hyperparameters, active extensions, commit SHA, completeness |
| `summary_mcq.json` | `summarize.py` output — what the dashboard reads |
| `rose_mcq.csv` / `rose_mcq_per_path.csv` | `export_csv.py` — per question / per reasoning path |
| `mcq_diagnostics.png` | learning curve + entropy calibration |

---
## Cell 1 — CONFIG · **the only cell you edit**

In [ ]:
from pathlib import Path
import os, sys

# Kaggle's log pane shows whatever has been flushed, and rose_chartqapro.py's
# own prints do not pass flush=True. Line-buffering stdout once here makes every
# downstream print stream out instead of arriving in blocks.
try:
    sys.stdout.reconfigure(line_buffering=True)
except (AttributeError, ValueError):
    pass

# ── where the pipeline code comes from ───────────────────────────
REPO_URL   = "https://github.com/ktahsinr/RoSE.git"
REPO_REF   = "fix/scoring-retrieval-and-extensions"   # branch, tag or SHA
LOCAL_CODE = None      # set to a /kaggle/input/... path holding chartqapro/*.py
                       # if the notebook must run with Internet OFF

# ── what to run ──────────────────────────────────────────────────
SPLIT        = "test"          # ChartQAPro split
LIMIT        = None            # None = ALL MCQ questions (this is the point)
SHARD, SHARDS = 0, 1           # split the set across parallel commits: SHARD=1 SHARDS=3

# ── model + RoSE hyperparameters ─────────────────────────────────
MODEL_NAME       = "Qwen/Qwen2.5-VL-7B-Instruct"
M_PATHS          = 5      # paper: 20 — 5 is the T4 compromise
K_DEMONSTRATIONS = 3
LAMBDA           = 1.2
TEMPERATURE      = 0.8
MAX_NEW_TOKENS   = 300
VISUAL_ALPHA     = 0.35

# ── extensions (see README table) ────────────────────────────────
PAPER_FAITHFUL      = False    # True = disable all ten, for the baseline row
EXTENSION_OVERRIDES = {}       # e.g. {"mcq_permute_options": False} for one ablation run

# ── headless safety ──────────────────────────────────────────────
TIME_BUDGET_MIN  = 11 * 60     # stop + save before Kaggle's 12h commit wall
CHECKPOINT_EVERY = 10
RESUME_FROM      = None        # path to a previous commit's checkpoint_mcq.json

# ── paths ────────────────────────────────────────────────────────
ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"   # not published
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"      # regenerated each run; never counts against output quota
RESULTS_DIR = WORK / "results"
DATASET_JSON    = SCRATCH / "chartqapro_mcq.json"
CHECKPOINT_FILE = RESULTS_DIR / "checkpoint_mcq.json"
FINAL_FILE      = RESULTS_DIR / "rose_mcq.json"      # summarize.py maps this to meta_mcq.json
META_FILE       = RESULTS_DIR / "meta_mcq.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"on kaggle    : {ON_KAGGLE}")
print(f"results dir  : {RESULTS_DIR}")
print(f"time budget  : {TIME_BUDGET_MIN} min")
print(f"resume from  : {RESUME_FROM}")

## Cell 2 — Dependencies + GPU

`Qwen2_5_VLForConditionalGeneration` only exists in `transformers >= 4.49`; on older
versions `rose_chartqapro.py` silently falls back to the Qwen2-VL class, which
mis-handles some 2.5 checkpoints. The install below is therefore version-checked,
not just presence-checked.

In [ ]:
import importlib, subprocess, sys

need = []
try:
    from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
except Exception:
    need.append("transformers==4.51.3")
for pkg, spec in [("accelerate", "accelerate>=0.34"),
                  ("bitsandbytes", "bitsandbytes>=0.43"),
                  ("sentence_transformers", "sentence-transformers>=3.0"),
                  ("datasets", "datasets>=2.19")]:
    if importlib.util.find_spec(pkg) is None:
        need.append(spec)

if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)
    print("✓ installed — restart is NOT required on Kaggle images")
else:
    print("✓ all dependencies present")

import torch, transformers
print(f"\ntorch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}        : {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    print("⚠  NO GPU — inference cells will fail. Set Accelerator to GPU T4 x2.")

## Cell 3 — Get the pipeline code

The RoSE algorithm, the prompts and every scoring rule live in the repo. Cloning
them (rather than pasting them here) is what keeps this notebook's numbers identical
to `python ablate.py` on a laptop. The commit SHA is recorded in `meta_mcq.json`.

In [ ]:
import subprocess, sys, shutil, importlib

def _sh(*args, cwd=None):
    return subprocess.run(args, cwd=cwd, capture_output=True, text=True)

CQA_DIR = None

if LOCAL_CODE:                                   # Internet OFF path
    src = Path(LOCAL_CODE)
    CQA_DIR = src if (src / "rose_chartqapro.py").exists() else src / "chartqapro"
    COMMIT_SHA = "local-copy"
else:
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    r = _sh("git", "clone", "--depth", "50", "--branch", REPO_REF, REPO_URL, str(CODE_DIR))
    if r.returncode != 0:                        # REPO_REF may be a bare SHA
        _sh("git", "clone", "--depth", "50", REPO_URL, str(CODE_DIR))
        _sh("git", "checkout", REPO_REF, cwd=str(CODE_DIR))
    CQA_DIR = CODE_DIR / "chartqapro"
    COMMIT_SHA = _sh("git", "rev-parse", "HEAD", cwd=str(CODE_DIR)).stdout.strip() or "unknown"

assert CQA_DIR and (CQA_DIR / "rose_chartqapro.py").exists(), (
    f"pipeline code not found at {CQA_DIR}. Either enable Internet (to clone "
    f"{REPO_URL}) or set LOCAL_CODE to a Kaggle dataset holding chartqapro/*.py")

sys.path.insert(0, str(CQA_DIR))

# The clone can only contain what was COMMITTED AND PUSHED to REPO_REF. A file
# that is merely present in the local working tree is not here — so the imports
# are split into what the run needs and what only the evaluation needs, and the
# gap is reported now rather than after ten GPU-hours.
REQUIRED = ["rose_chartqapro", "scoring"]            # inference + verdicts
OPTIONAL = ["ablate", "summarize", "export_csv"]     # offline evaluation only

mods, missing = {}, []
for name in REQUIRED + OPTIONAL:
    try:
        mods[name] = importlib.import_module(name)
    except Exception as exc:
        mods[name] = None
        missing.append((name, exc))

rose, scoring = mods["rose_chartqapro"], mods["scoring"]
ablate, summarize, export_csv = mods["ablate"], mods["summarize"], mods["export_csv"]

assert rose and scoring, (
    f"the run itself cannot start: {[n for n, _ in missing if n in REQUIRED]} "
    f"missing from {CQA_DIR}")

# summarize.py does `from ablate import ablation_table`; an older ablate.py
# without it breaks both.
STALE_ABLATE = ablate is not None and not hasattr(ablate, "ablation_table")

print(f"✓ pipeline   : {CQA_DIR}")
print(f"✓ commit     : {COMMIT_SHA[:12]}  ({REPO_REF})")
print(f"✓ required   : rose_chartqapro, scoring")
print(f"  evaluation : " + ", ".join(n for n in OPTIONAL if mods[n]) or "  evaluation : none")

if missing or STALE_ABLATE:
    print("\n" + "!" * 70)
    for n, exc in missing:
        print(f"  missing: {n}.py  ({type(exc).__name__})")
    if STALE_ABLATE:
        print("  stale  : ablate.py has no ablation_table() — Cells 10-12 need the newer one")
    print(f"\n  These files exist in your working tree but are NOT on '{REPO_REF}'.")
    print("  Inference is unaffected; the offline evaluation cells will skip.")
    print("  To fix, from the repo root:\n")
    print("      git add chartqapro/summarize.py chartqapro/ablate.py")
    print('      git commit -m "Add summarize.py + ablation_table for the Kaggle run"')
    print(f"      git push origin {REPO_REF}\n")
    print("  then re-run this cell. Or set LOCAL_CODE to a Kaggle dataset holding")
    print("  chartqapro/*.py uploaded straight from your laptop.")
    print("!" * 70)

## Cell 4 — Build the MCQ set

Two things the pipeline needs that the raw dataset does not hand over:

1. **`choices` as a list.** ChartQAPro carries MCQ options inline in the question
   text far more often than in a column, so options are recovered by column first,
   then by parsing the question.
2. **Options removed from the question stem.** `mcq_permute_options` rotates the
   option order across the *m* paths. If the original `A) … B) …` text were left in
   the stem, the model would see two contradictory letterings and the permutation
   would inject noise instead of cancelling position bias.

Rows where fewer than two options can be recovered are kept but flagged — they run
as open-ended questions and are counted in the report below.

In [ ]:
import ast, io, json, re
from collections import Counter
from PIL import Image

MCQ_LABELS     = {"multi choice", "multiple choice", "multi-choice", "mcq"}
OPTION_COLUMNS = ("Choices", "Options", "Answer Options", "Choice List", "choices", "options")


def _as_list(v):
    """Column value -> list of option strings, or None."""
    if v is None:
        return None
    if isinstance(v, (list, tuple)):
        out = [str(x).strip() for x in v if str(x).strip()]
        return out if len(out) >= 2 else None
    s = str(v).strip()
    if not s:
        return None
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                out = [str(x).strip() for x in p if str(x).strip()]
                return out if len(out) >= 2 else None
        except Exception:
            pass
    parts = [p.strip() for p in re.split(r"\s*\|\s*|\s*;\s*", s) if p.strip()]
    return parts if len(parts) >= 2 else None


_LETTER = re.compile(r"\(?\b([A-Ea-e])\s*[\)\.\:]\s+")


def _split_inline(question):
    """'... ? A) 12  B) 45' -> ('... ?', ['12', '45']).  Returns (q, None) if absent."""
    q = str(question)
    run, expected = [], 0
    for m in _LETTER.finditer(q):
        if m.group(1).lower() == chr(ord("a") + expected):   # must run A, B, C, ...
            run.append(m)
            expected += 1
    if len(run) < 2:
        return q, None
    stem = q[:run[0].start()].strip()
    opts = []
    for i, m in enumerate(run):
        end = run[i + 1].start() if i + 1 < len(run) else len(q)
        opts.append(q[m.end():end].strip(" \n\t.,;"))
    if not stem or not all(opts):
        return q, None
    stem = re.sub(r"(?:options?|choices?|choose from)\s*:?\s*$", "", stem, flags=re.I).strip()
    return stem, opts


def _split_block(question):
    """'... ?\nOptions:\n12\n45' -> ('... ?', ['12', '45'])."""
    q = str(question)
    m = re.search(r"\n?\s*(?:options?|choices?)\s*:\s*", q, re.I)
    if not m:
        return q, None
    stem, tail = q[:m.start()].strip(), q[m.end():]
    opts = [o.strip(" -•\t") for o in re.split(r"\n|\s*\|\s*", tail) if o.strip(" -•\t")]
    if not stem or len(opts) < 2:
        return q, None
    return stem, opts


def extract_options(row):
    """(stem, options, source). options is None when nothing parseable was found."""
    for col in OPTION_COLUMNS:
        opts = _as_list(row.get(col))
        if opts:
            stem, _ = _split_inline(row.get("Question", ""))
            return stem, opts, f"column:{col}"
    stem, opts = _split_inline(row.get("Question", ""))
    if opts:
        return stem, opts, "inline"
    stem, opts = _split_block(row.get("Question", ""))
    if opts:
        return stem, opts, "block"
    return str(row.get("Question", "")).strip(), None, "none"


def _gold(v):
    if isinstance(v, (list, tuple)):
        return [str(x) for x in v]
    s = str(v).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x) for x in p]
        except Exception:
            pass
    return s


def _save_image(field, path):
    if isinstance(field, dict) and field.get("bytes"):
        img = Image.open(io.BytesIO(field["bytes"]))
    elif isinstance(field, (bytes, bytearray)):
        img = Image.open(io.BytesIO(field))
    elif isinstance(field, Image.Image):
        img = field
    else:                                    # already a path on disk
        return str(field)
    img.convert("RGB").save(path, format="PNG")
    return str(path)


# ── load + filter ────────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset("ahmed-masry/ChartQAPro", split=SPLIT)
print(f"{SPLIT} split: {len(ds)} questions")
print("by type:", dict(Counter(str(t) for t in ds["Question Type"]).most_common()))

# Select the MCQ rows by column first, so only the images this run actually
# needs get decoded and written — the other ~80% of the split is never touched.
qtypes  = [str(t).strip().lower() for t in ds["Question Type"]]
mcq_idx = [i for i, t in enumerate(qtypes) if t in MCQ_LABELS]
if SHARDS > 1:
    mcq_idx = mcq_idx[SHARD::SHARDS]
if LIMIT:
    mcq_idx = mcq_idx[:LIMIT]
print(f"MCQ rows selected: {len(mcq_idx)}")

records, sources = [], Counter()
for idx, i in enumerate(mcq_idx):
    row = ds[i]
    stem, opts, src = extract_options(row)
    sources[src] += 1
    records.append({
        "id":            str(row.get("id", f"mcq-{idx:05d}")),
        "question":      stem,
        "choices":       opts,
        "answer":        _gold(row.get("Answer")),
        "question_type": "mcq",                       # the label the pipeline expects
        "image":         _save_image(row.get("image"), IMAGES_DIR / f"{idx:05d}.png"),
        "orig_question": str(row.get("Question", "")),
        "orig_type":     str(row.get("Question Type", "")),
        "year":          row.get("Year"),
    })

DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))

print(f"\nMCQ questions to run : {len(records)}")
print(f"option source        : {dict(sources)}")
n_missing = sum(1 for r in records if not r["choices"])
print(f"no options recovered : {n_missing}  ({n_missing/max(len(records),1)*100:.1f}%)"
      f"  — these run as open-ended")
print(f"dataset json         : {DATASET_JSON}")
print(f"images               : {IMAGES_DIR}")

## Cell 5 — Pre-flight check

Two failure modes are silent and expensive if they only surface after a 10-hour run:
options parsed wrong, and gold answers that no longer resolve to any option. Both are
checked here with the same `scoring.resolve_choice` the scorer uses.

In [ ]:
resolvable = 0
for r in records:
    if not r["choices"]:
        continue
    golds = scoring.split_gold(r["answer"])
    if any(scoring.resolve_choice(g, r["choices"]) is not None for g in golds):
        resolvable += 1
n_with = sum(1 for r in records if r["choices"])

print(f"gold resolves to an option : {resolvable}/{n_with} "
      f"({resolvable/max(n_with,1)*100:.1f}% of rows that have options)")
print("   (unresolvable gold still scores, via normalized text match — but a low "
      "number here means the option parse is wrong)\n")

for r in records[:3]:
    print("=" * 70)
    print("raw      :", r["orig_question"][:200].replace("\n", " "))
    print("stem     :", r["question"][:200].replace("\n", " "))
    print("choices  :", r["choices"])
    print("gold     :", r["answer"])
    print("image    :", r["image"])

## Cell 6 — Point the pipeline at this run

`rose_chartqapro.py` reads its configuration from module-level constants, so a run is
configured by overriding them — nothing is forked or copied.

In [ ]:
rose.MODEL_NAME       = MODEL_NAME
rose.TARGET_TYPES     = {"mcq"}
rose.M_PATHS          = M_PATHS
rose.K_DEMONSTRATIONS = K_DEMONSTRATIONS
rose.LAMBDA           = LAMBDA
rose.TEMPERATURE      = TEMPERATURE
rose.MAX_NEW_TOKENS   = MAX_NEW_TOKENS
rose.VISUAL_ALPHA     = VISUAL_ALPHA
rose.CHECKPOINT_EVERY = CHECKPOINT_EVERY

rose.DATASET_PATH     = str(DATASET_JSON)
rose.IMAGES_DIR       = str(IMAGES_DIR)
rose.RESULTS_DIR      = str(RESULTS_DIR)
rose.CHECKPOINT_FILE  = str(CHECKPOINT_FILE)
rose.FINAL_FILE       = str(FINAL_FILE)

rose.PAPER_FAITHFUL   = PAPER_FAITHFUL
rose.EXTENSIONS.update(EXTENSION_OVERRIDES)

print(f"model   : {rose.MODEL_NAME}")
print(f"types   : {sorted(rose.TARGET_TYPES)}")
print(f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  T={rose.TEMPERATURE}")
print(f"paper faithful : {rose.PAPER_FAITHFUL}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

## Cell 7 — The headless loop

Same body as `rose_chartqapro.run()` — same `process_one`, same warm-restart pool
rebuild, same checkpoint semantics — with three things a Kaggle commit needs:

- a **wall-clock budget** that stops cleanly instead of being killed mid-write,
- **one log line per question** (a 12h commit log stays readable),
- `meta_mcq.json` carrying **`complete`**, the commit SHA and the elapsed time, so a
  partial run can never be mistaken for a finished one.

In [ ]:
import time, traceback, json


def run_mcq_headless(data, model, processor, embedder, clip_model, clip_processor):
    t0 = time.time()
    budget_s = TIME_BUDGET_MIN * 60

    # ── resume ───────────────────────────────────────────────────
    results = []
    resume_src = None
    for cand in (CHECKPOINT_FILE, Path(RESUME_FROM) if RESUME_FROM else None):
        if cand and Path(cand).exists():
            results = json.loads(Path(cand).read_text())
            resume_src = cand
            break
    start_idx = len(results)
    if resume_src:
        print(f"↩  resuming at {start_idx}/{len(data)} from {resume_src}")

    # ── rebuild the pool from what is already answered ───────────
    pool = rose.ExperiencePool(embedder)
    for r in results:
        if r.get("method") == "error":
            continue
        pool.add(question=r["question"],
                 rationale=r.get("best_rationale") or r.get("prediction", ""),
                 answer=r.get("prediction", ""),
                 qtype=r.get("question_type", "mcq"),
                 uncertainty=r.get("uncertainty", 0.5),
                 complexity=r.get("complexity", 1.0),
                 clip_emb=None)          # embeddings are not checkpointed; text-only retrieval

    def save(final=False):
        CHECKPOINT_FILE.write_text(json.dumps(results, indent=2))
        if final:
            FINAL_FILE.write_text(json.dumps(results, indent=2))

    print(f"\n{'='*66}\n  RoSE · ChartQAPro MCQ · {len(data)-start_idx} to go"
          f"  |  m={rose.M_PATHS} k={rose.K_DEMONSTRATIONS}\n{'='*66}")

    complete = True
    try:
        for i, sample in enumerate(data[start_idx:], start=start_idx):
            if time.time() - t0 > budget_s:
                print(f"\n⏱  time budget reached at {i}/{len(data)} — saving and stopping cleanly")
                complete = False
                break
            try:
                r = rose.process_one(sample, model, processor, pool,
                                     clip_model=clip_model, clip_processor=clip_processor)
                results.append(r)
                print(f"[{i+1:04d}/{len(data)}] {'✓' if r['is_correct'] else '✗'} "
                      f"{r['method'][:9]:<9} pool={r['pool_size']:<4} "
                      f"pred={str(r['prediction'])[:22]:<22} gold={str(r['ground_truth'])[:22]:<22} "
                      f"u={r['uncertainty']:.2f}", flush=True)
            except Exception as exc:
                print(f"[{i+1:04d}/{len(data)}] ⚠  {exc}", flush=True)
                traceback.print_exc()
                results.append({"id": sample.get("id", ""), "question": sample.get("question", ""),
                                "question_type": "mcq", "ground_truth": sample.get("answer", ""),
                                "prediction": "ERROR", "is_correct": False,
                                "error": str(exc), "method": "error"})

            if (i + 1) % CHECKPOINT_EVERY == 0:
                save()
                ok = [x for x in results if x.get("method") != "error"]
                acc = sum(x.get("is_correct", False) for x in ok) / max(len(ok), 1) * 100
                print(f"  💾 checkpoint {i+1}/{len(data)}  running acc={acc:.1f}%  "
                      f"elapsed={(time.time()-t0)/60:.0f}m", flush=True)
    except KeyboardInterrupt:
        # Ctrl-C (and Kaggle's stop button) raises BaseException, which
        # the per-question handler deliberately does not catch. Without
        # this the function would unwind before writing its final files.
        print("\n⛔ interrupted — saving what is already answered", flush=True)
        complete = False

    save(final=True)

    META_FILE.write_text(json.dumps({
        "model": rose.MODEL_NAME, "embed_model": rose.EMBED_MODEL,
        "clip_model": "openai/clip-vit-base-patch32" if clip_model else None,
        "m_paths": rose.M_PATHS, "k_demonstrations": rose.K_DEMONSTRATIONS,
        "lambda": rose.LAMBDA, "temperature": rose.TEMPERATURE,
        "visual_alpha": rose.VISUAL_ALPHA, "max_new_tokens": rose.MAX_NEW_TOKENS,
        "numeric_tolerance": rose.NUMERIC_TOLERANCE,
        "offer_unanswerable": rose.OFFER_UNANSWERABLE,
        "target_types": ["mcq"], "paper_faithful": rose.PAPER_FAITHFUL,
        "extensions": rose.active_extensions(),
        "dataset": f"ahmed-masry/ChartQAPro:{SPLIT}", "shard": [SHARD, SHARDS],
        "code_commit": COMMIT_SHA, "code_ref": REPO_REF,
        "n_target": len(data), "n_results": len(results),
        "complete": complete and len(results) >= len(data),
        "elapsed_min": round((time.time() - t0) / 60, 1),
        "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, indent=2))

    print(f"\n{'✓ COMPLETE' if complete else '◐ PARTIAL — rerun with RESUME_FROM'}"
          f"  {len(results)}/{len(data)}  →  {FINAL_FILE}")
    return results

## Cell 8 — Run it  ⏳

The long cell. Loads the VLM (4-bit), the sentence embedder and CLIP, then streams
every MCQ question through RoSE with the experience pool growing as it goes.

**Expect a long silence before the first line.** On a fresh session the model
weights are downloaded (several GB) before anything is printed, and the first
question then needs `M_PATHS + 1` generations. Minutes without output here is
normal; the per-question lines start once loading finishes.

**In a Save & Run All commit, Kaggle does not render cell output live** — the
notebook you see updates only when the version finishes. To watch a commit in
flight, open the version and read the **Logs** pane. Interactive sessions do
stream the output into the cell.


In [ ]:
import time

data = json.loads(DATASET_JSON.read_text())
print(f"[{time.strftime('%H:%M:%S')}] {len(data)} MCQ questions queued", flush=True)
print(f"[{time.strftime('%H:%M:%S')}] loading {rose.MODEL_NAME} — a fresh session "
      f"downloads the weights first, so this step is silent for several minutes",
      flush=True)

model, processor, embedder = rose.load_models()
clip_model, clip_processor = rose.load_clip()
print(f"[{time.strftime('%H:%M:%S')}] models ready — starting the sweep", flush=True)

results = run_mcq_headless(data, model, processor, embedder, clip_model, clip_processor)

---
# Evaluation

Everything below is CPU-only and reads `rose_mcq.json` from disk, so it can be rerun
on a laptop — or on a partial run, mid-sweep. Each cell no-ops with a message rather
than raising, so a commit still publishes its outputs when the run was cut short.

## Cell 9 — Headline accuracy

In [ ]:
import json
from pathlib import Path

# FINAL_FILE appears only once the sweep returns. A run that is still
# going — or was interrupted — has everything answered so far in the
# checkpoint, so evaluate whichever exists and say which one was used.
SOURCE = FINAL_FILE if Path(FINAL_FILE).exists() else CHECKPOINT_FILE

rows = json.loads(Path(SOURCE).read_text()) if Path(SOURCE).exists() else []
meta = json.loads(Path(META_FILE).read_text()) if Path(META_FILE).exists() else {}

if not rows:
    print(f"no results yet — run Cell 8, or wait for its first checkpoint "
          f"(written every {CHECKPOINT_EVERY} questions)")
else:
    partial = SOURCE != FINAL_FILE
    print(f"source : {Path(SOURCE).name}"
          + ("   ← PARTIAL, straight from the checkpoint" if partial else ""))
    print(f"rows   : {len(rows)}")
    if meta:
        print(f"run    : {meta.get('n_results')}/{meta.get('n_target')}   "
              f"complete={meta.get('complete')}   "
              f"commit={str(meta.get('code_commit'))[:12]}")
    rose.print_results_table(rows)


## Cell 10 — Offline aggregation ablation

`numeric_vote_clustering`, `drop_malformed_paths` and `confidence_weighted_vote` only
change how the *m* logged per-path answers are combined, so all four configurations
are recomputed from this one run — no GPU. The other seven extensions change the
prompt, the decoding or the pool, so each needs its own commit with the flag flipped
in `EXTENSION_OVERRIDES`.

In [ ]:
if not rows:
    print("no results yet")
elif ablate is None or STALE_ABLATE:
    print("skipped — ablate.ablation_table() is not in this clone (see Cell 3).\n"
          "The per-path logs are in rose_mcq.json, so this runs later on a laptop:\n"
          f"    python ablate.py {Path(SOURCE).name} --by-type")
else:
    for block in ablate.ablation_table(rows, by_type=True):
        print(f"\n{'='*66}\n  {block['label']}   (n={block['n']})\n{'='*66}")
        for c in block["configs"]:
            d = f"{c['delta']:+.1f}" if c.get("delta") is not None else "  base"
            print(f"  {c['name']:<34} {c['acc']:5.1f}%  ({c['correct']}/{c['total']})  {d}")
        if block.get("skipped"):
            print(f"  ({block['skipped']} rows had no per-path log and used their stored prediction)")

## Cell 11 — `summarize.py` + CSV export

`summary_mcq.json` is exactly what the web dashboard reads, and the CSVs are what goes
into a results table. Both re-score every row through `scoring.py` on the way out, so
they carry the current verdict rather than whatever flag was written during the run.

In [ ]:
import subprocess, sys

summary = None
if not rows:
    print("no results yet")
elif summarize is None or STALE_ABLATE:
    print("skipped — summarize.py is not in this clone (see Cell 3).\n"
          f"    python summarize.py {Path(SOURCE).name}      # later, on a laptop")
else:
    summary = summarize.summarize(str(SOURCE))
    (RESULTS_DIR / "summary_mcq.json").write_text(json.dumps(summary, indent=2))

    o = summary["overall"]
    print(f"overall (rescored) : {o['rescored']['acc']:.1f}%  "
          f"({o['rescored']['correct']}/{o['rescored']['total']})")
    print(f"overall (as run)   : {o['run']['acc']:.1f}%   drift={o['drift']:+d}")
    print(f"errors             : {summary['n_errors']}")
    print("\nby method:")
    for b in summary["by_method"]:
        print(f"   {b['key']:<16} {b['acc']:5.1f}%  ({b['correct']}/{b['total']})")
    print("\nby vote mode:")
    for b in summary["by_vote_mode"]:
        print(f"   {b['key']:<16} {b['acc']:5.1f}%  ({b['correct']}/{b['total']})")
    print("\nanswerability confusion:", summary["confusion"])

# export_csv.py stands alone — it only needs scoring.py, so the CSVs are written
# even when the summarize/ablate pair is missing from the clone.
if rows and (CQA_DIR / "export_csv.py").exists():
    # --per-path writes both the per-question CSV and the per-path one
    r = subprocess.run([sys.executable, str(CQA_DIR / "export_csv.py"),
                        str(SOURCE), "--per-path"], capture_output=True, text=True)
    print("\n" + (r.stdout.strip() or r.stderr.strip()))

## Cell 12 — Does the pool actually help?

The RoSE claim in two plots: accuracy should climb as the experience pool grows, and
the entropy the pool is filtered by should be predictive of correctness — high
uncertainty bins should score worse than low ones. If the second plot is flat, λ is
gating on noise.

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
BLUE             = "#2a78d6"

if not rows or summary is None:
    print("skipped — needs the summary from Cell 11")
else:
    curve = summary["curve"]
    bins  = summary["uncertainty_bins"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), facecolor="white")

    # ── learning curve ───────────────────────────────────────────
    xs = [p["i"] for p in curve["points"]]
    ys = [p["roll_acc"] for p in curve["points"]]
    ax1.plot(xs, ys, color=BLUE, lw=2, solid_capstyle="round")
    if curve["phase_switch"]:
        ax1.axvspan(0, curve["phase_switch"], color=GRID, alpha=0.7, lw=0)
        ax1.annotate("zero-shot (pool < k)", xy=(curve["phase_switch"], 1),
                     xycoords=("data", "axes fraction"), xytext=(4, -10),
                     textcoords="offset points", fontsize=8.5, color=MUTED, va="top")
    final = summary["overall"]["rescored"]["acc"]
    ax1.axhline(final, color=MUTED, lw=1, ls=(0, (4, 3)))
    ax1.annotate(f"{final:.1f}% overall", xy=(1, final), xycoords=("axes fraction", "data"),
                 xytext=(-4, 4), textcoords="offset points",
                 ha="right", va="bottom", fontsize=8.5, color=MUTED)
    ax1.set_title(f"Rolling accuracy as the pool grows  (window {curve['window']})",
                  fontsize=11, color=INK, loc="left")
    ax1.set_xlabel("question, in stream order", fontsize=9, color=MUTED)
    ax1.set_ylabel("accuracy %", fontsize=9, color=MUTED)

    # ── uncertainty calibration ──────────────────────────────────
    labels = [f"{b['lo']:.2f}–{b['hi']:.2f}" for b in bins]
    accs   = [b["acc"] for b in bins]
    ax2.bar(range(len(bins)), accs, color=BLUE, width=0.62)
    for i, b in enumerate(bins):
        ax2.text(i, b["acc"] + 2, f"{b['acc']:.0f}%\nn={b['total']}",
                 ha="center", va="bottom", fontsize=8.5, color=MUTED)
    ax2.set_xticks(range(len(bins)))
    ax2.set_xticklabels(labels, fontsize=8.5, color=MUTED)
    ax2.set_yticks([0, 25, 50, 75, 100])
    ax2.set_title("Accuracy by entropy bin  (low uncertainty → left)",
                  fontsize=11, color=INK, loc="left")
    ax2.set_xlabel("path-vote entropy", fontsize=9, color=MUTED)
    ax2.set_ylabel("accuracy %", fontsize=9, color=MUTED)
    ax2.set_ylim(0, 118)

    for ax in (ax1, ax2):
        ax.grid(axis="y", color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=9)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "mcq_diagnostics.png", dpi=150, facecolor="white")
    plt.show()

## Cell 13 — MCQ-specific diagnostics

Position bias is the failure mode `mcq_permute_options` exists to kill: if the
predicted-letter distribution is much flatter or much more skewed than the gold
distribution, the rotation is doing (or failing to do) its job, and this table is the
evidence for that claim in the write-up.

In [ ]:
from collections import Counter

if not rows:
    print("no results yet")
else:
    valid = [r for r in rows if r.get("method") != "error" and r.get("options_shown")]
    pred_letters, gold_letters = Counter(), Counter()
    for r in valid:
        opts = r["options_shown"]
        pi = scoring.resolve_choice(r.get("prediction", ""), opts)
        gi = next((scoring.resolve_choice(g, opts) for g in scoring.split_gold(r.get("ground_truth"))
                   if scoring.resolve_choice(g, opts) is not None), None)
        pred_letters[chr(65 + pi) if pi is not None else "?"] += 1
        gold_letters[chr(65 + gi) if gi is not None else "?"] += 1

    n = max(len(valid), 1)
    print(f"canonical option distribution over {len(valid)} MCQ rows\n")
    print(f"  {'letter':<8}{'predicted':>12}{'gold':>10}{'skew':>10}")
    for L in sorted(set(pred_letters) | set(gold_letters)):
        p, g = pred_letters[L], gold_letters[L]
        print(f"  {L:<8}{p:>7} {p/n*100:4.0f}%{g:>6} {g/n*100:4.0f}%{(p-g)/n*100:>9.1f}pp")
    print(f"\n  '?' = did not resolve to any option "
          f"(pred {pred_letters['?']}, gold {gold_letters['?']})")

    agree = [r.get("agreement") for r in valid if r.get("agreement") is not None]
    if agree:
        print(f"\nmean path agreement : {sum(agree)/len(agree):.3f}")
    unanimous = [r for r in valid if (r.get("agreement") or 0) >= 0.999]
    if unanimous:
        acc_u = sum(r["is_correct"] for r in unanimous) / len(unanimous) * 100
        print(f"unanimous questions : {len(unanimous)}  ({acc_u:.1f}% correct)")
    split = [r for r in valid if (r.get("agreement") or 0) < 0.6]
    if split:
        acc_s = sum(r["is_correct"] for r in split) / len(split) * 100
        print(f"split questions     : {len(split)}  ({acc_s:.1f}% correct)")

    wrong = [r for r in valid if not r.get("is_correct")][:10]
    print(f"\nfirst {len(wrong)} misses:")
    for r in wrong:
        print(f"  · {str(r['question'])[:70]:<70} pred={str(r['prediction'])[:14]:<14} "
              f"gold={str(r['ground_truth'])[:14]}")

## Cell 14 — Outputs

Download `results/` from the version's **Output** tab. Dropping those files into
`chartqapro/results/` in the repo makes them show up in the **ChartQAPro** tab of the
web app (`localhost:5173/#cqa`) — the dashboard reads `summarize.py`, which is why its
numbers match Cells 9–11 exactly.

In [ ]:
print(f"{RESULTS_DIR}\n")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size/1024:9.1f} KB")

if meta and not meta.get("complete"):
    print("\n◐  PARTIAL RUN — to continue, add this version's output as a data source and set:")
    print(f'    RESUME_FROM = "/kaggle/input/<notebook-output>/results/checkpoint_mcq.json"')